# Interventions & Potential Outcomes

Companion notebook for the [Interventions & Potential Outcomes lesson](https://ml-viz.vercel.app/courses/causal-inference/02-interventions-and-potential-outcomes).

Because we can simulate, we know each unit's BOTH potential outcomes — so we can compute the true
**ATE** and watch the **naive estimate** miss it, **backdoor adjustment** recover it, and
**randomization** make adjustment unnecessary. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## 1 — A simulation with known potential outcomes

A confounder Z affects treatment and outcome. The treatment has a TRUE effect of +0.2. We generate
both Y(0) and Y(1) for every unit (only possible in simulation) so we know the ground-truth ATE.

In [ ]:
n = 20000
Z = rng.random(n)                                   # confounder (e.g. severity)
TRUE_EFFECT = 0.2
Y0 = 0.3 + 0.5 * Z + rng.normal(0, 0.05, n)         # outcome if untreated
Y1 = Y0 + TRUE_EFFECT                                # outcome if treated (constant effect)
ate_true = (Y1 - Y0).mean()
print(f'TRUE ATE (we can compute it because we simulated both potential outcomes): {ate_true:.3f}')

## 2 — Confounded assignment → biased naive estimate

Higher-Z units are treated more often. We only get to observe the potential outcome matching each
unit's actual treatment — and the naive difference is biased upward by the confounding.

In [ ]:
T = (rng.random(n) < Z).astype(int)                 # confounded: treatment depends on Z
Y = np.where(T == 1, Y1, Y0)                        # observe only the realized outcome

naive = Y[T==1].mean() - Y[T==0].mean()
print(f'naive estimate: {naive:.3f}  (biased: true is {ate_true:.3f})')
print(f'confounding bias: {naive - ate_true:+.3f}')

## 3 — Backdoor adjustment recovers the ATE

Estimate the effect within strata of the confounder Z, then average over Z's distribution — the
backdoor formula. Since Z is the only confounder, this recovers the true effect.

In [ ]:
def backdoor_adjust(Z, T, Y, bins=20):
    edges = np.linspace(0, 1, bins + 1)
    effs, wts = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (Z >= lo) & (Z < hi)
        if (T[m]==1).sum() and (T[m]==0).sum():
            effs.append(Y[m & (T==1)].mean() - Y[m & (T==0)].mean())
            wts.append(m.sum())
    return np.average(effs, weights=wts)

print(f'backdoor-adjusted estimate: {backdoor_adjust(Z, T, Y):.3f}  (recovers true {ate_true:.3f})')

## 4 — Randomization makes adjustment unnecessary

If instead we *randomize* treatment (ignoring Z), the groups are comparable by construction and the
naive estimate is already unbiased — this is why RCTs and A/B tests are the gold standard.

In [ ]:
T_rand = (rng.random(n) < 0.5).astype(int)          # do(T): assignment independent of Z
Y_rand = np.where(T_rand == 1, Y1, Y0)
print(f'naive estimate under randomization: {Y_rand[T_rand==1].mean() - Y_rand[T_rand==0].mean():.3f}')
print(f'(already unbiased — true {ate_true:.3f} — no adjustment needed)')

## ✏️ Your turn

**Exercise.** Implement `ate(Y1, Y0)` (the average treatment effect from both potential outcomes) and
`confounding_bias(naive, true_ate)` (how far the naive estimate is from the truth). Then you'll have
the full decomposition: naive = true ATE + bias.

In [ ]:
def ate(Y1, Y0):
    # TODO(you): average of the individual treatment effects Y1 - Y0
    return ...

def confounding_bias(naive, true_ate):
    # TODO(you): the gap between the naive estimate and the true ATE
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(ate(Y1, Y0), TRUE_EFFECT, atol=1e-9)
b = confounding_bias(naive, ate(Y1, Y0))
assert b > 0                                          # confounding inflated the naive estimate
assert np.isclose(naive, ate(Y1, Y0) + b)             # naive = true ATE + bias
# adjustment removes most of the bias
assert abs(backdoor_adjust(Z, T, Y) - ate(Y1, Y0)) < abs(b)
print(f'\u2713 ATE={ate(Y1,Y0):.3f}, naive bias={b:+.3f}, adjustment recovers the truth')

<details>
<summary>Solution</summary>

```python
def ate(Y1, Y0):
    return (Y1 - Y0).mean()

def confounding_bias(naive, true_ate):
    return naive - true_ate
```

In real data you never see both Y1 and Y0, so you can't compute the ATE directly — you estimate it
by randomization or by adjusting for confounders. The simulation lets us cheat and verify the
estimators actually recover the known truth.

</details>